# Modelo matemático para la pastelería Vainilla Atelier
**Investigación de Operaciones 1, UAO.** Emily Yanguas · Juan Camilo Sossa · Daniela Vásquez · Alejandro Muñoz · Isabella Morales

Modelo de programación lineal entera para decidir cuántas unidades, recetas, horneadas y cajitas de cada producto elaborar por semana para **maximizar la utilidad [$/sem]**.

## 0. Librerías

In [ ]:
# API y solucionador CBC compatibles con el modelo utilizado.
!pip -q install "pulp==3.3.0" pandas


In [ ]:
import pulp as pl
import pandas as pd

pd.options.display.float_format = '{:,.2f}'.format
pd.set_option("display.max_colwidth", None)

# Formato de pesos colombianos: $1.234.567,89
def pesos(valor):
    texto = f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return "$" + texto

## 1. Conjuntos e índices

In [ ]:
I = [1, 2, 3, 4, 5, 6, 7, 8]   # productos
K = [5, 6, 7]                  # productos que se venden en cajitas de 2 unidades

nombre = {1: "Pastel personal de zanahoria", 2: "Pastel personal forma de corazón",
          3: "Cuchareable de maracumango", 4: "Cuchareable de fresa",
          5: "Galleta de chocolate", 6: "Alfajor", 7: "Rollo de canela", 8: "Cuchareable de lulada"}

# Nombres cortos para las variables de PuLP
corto = {1: "Zanahoria", 2: "Corazon", 3: "Maracumango", 4: "Fresa",
         5: "Galleta", 6: "Alfajor", 7: "Rollo", 8: "Lulada"}

## 2. Parámetros

In [ ]:
U  = {1: 9523.64, 2: 8811.30, 3: 4575.90, 4: 5022.50,
      5: 1212.53, 6: 1585.64, 7: 1323.22, 8: 5022.50}           # utilidad [$/und]
D  = {1: 4, 2: 7, 3: 7, 4: 12, 5: 68, 6: 64, 7: 68, 8: 12}      # demanda [und/sem]
CH = {1: 4, 2: 4, 3: 20, 4: 20, 5: 25, 6: 30, 7: 10, 8: 20}     # capacidad por horneada [und/horneada]
TC = {1: 20, 2: 20, 3: 25, 4: 25, 5: 10, 6: 10, 7: 25, 8: 25}   # tiempo por horneada [min/horneada]
RR = {1: 2, 2: 5, 3: 10, 4: 10, 5: 94, 6: 50, 7: 26, 8: 10}     # rendimiento por receta [und/receta]
TR = {1: 120, 2: 120, 3: 180, 4: 180, 5: 180, 6: 180, 7: 180, 8: 180}  # mano de obra por receta [min-persona/receta]

UC = 2      # unidades por cajita [und/cajita]
H  = 1200   # horno disponible [min/sem]
MO = 2400   # mano de obra disponible [min-persona/sem]

df_parametros = pd.DataFrame({"Producto": nombre, "U ($/und)": U, "D (und/sem)": D,
                              "CH (und/horneada)": CH, "TC (min/horneada)": TC,
                              "RR (und/receta)": RR, "TR (min-persona/receta)": TR})
df_parametros.index.name = "i"
display(df_parametros)

,Producto,U ($/und),D (und/sem),CH (und/horneada),TC (min/horneada),RR (und/receta),TR (min-persona/receta)
i,,,,,,,
1,Pastel personal de zanahoria,"9,523.64",4,4,20,2,120
2,Pastel personal forma de corazón,"8,811.30",7,4,20,5,120
3,Cuchareable de maracumango,"4,575.90",7,20,25,10,180
4,Cuchareable de fresa,"5,022.50",12,20,25,10,180
5,Galleta de chocolate,"1,212.53",68,25,10,94,180
6,Alfajor,"1,585.64",64,30,10,50,180
7,Rollo de canela,"1,323.22",68,10,25,26,180
8,Cuchareable de lulada,"5,022.50",12,20,25,10,180


## 3. Variables de decisión
Todas son enteras y no negativas: $X_i$ [und/sem], $Y_i$ [horneadas/sem], $L_i$ [recetas/sem] y $C_k$ [cajitas/sem].

In [ ]:
modelo = pl.LpProblem("Vainilla_Atelier", pl.LpMaximize)

X = {i: pl.LpVariable(f"X{i}_{corto[i]}", lowBound=0, cat=pl.LpInteger) for i in I}  # unidades
Y = {i: pl.LpVariable(f"Y{i}_{corto[i]}", lowBound=0, cat=pl.LpInteger) for i in I}  # horneadas
L = {i: pl.LpVariable(f"L{i}_{corto[i]}", lowBound=0, cat=pl.LpInteger) for i in I}  # recetas
C = {k: pl.LpVariable(f"C{k}_{corto[k]}", lowBound=0, cat=pl.LpInteger) for k in K}  # cajitas

## 4. Función objetivo
$\text{Max } Z = \sum_{i \in I} U_i \cdot X_i \quad [\$/\text{sem}]$

In [ ]:
modelo += pl.lpSum(U[i] * X[i] for i in I), "Utilidad_total_semanal"

## 5. Restricciones

In [ ]:
# Se guarda cada restricción (grupo, nombre, lado izquierdo, sentido, límite, unidad) para el reporte
restricciones = []

def agregar(grupo, nombre_r, lado_izq, sentido, limite, unidad):
    if sentido == "<=":
        modelo.addConstraint(lado_izq <= limite, nombre_r)
    else:
        modelo.addConstraint(lado_izq == limite, nombre_r)
    restricciones.append((grupo, nombre_r, lado_izq, sentido, limite, unidad))

for i in I:   # Demanda: X_i <= D_i
    agregar("Demanda", f"Demanda_{corto[i]}", X[i], "<=", D[i], "und/sem")

for i in I:   # Capacidad por horneada: X_i <= CH_i * Y_i
    agregar("Capacidad por horneada", f"Capacidad_horneada_{corto[i]}", X[i], "<=", CH[i] * Y[i], "und/sem")

for i in I:   # Rendimiento por receta: X_i <= RR_i * L_i
    agregar("Rendimiento por receta", f"Rendimiento_receta_{corto[i]}", X[i], "<=", RR[i] * L[i], "und/sem")

for k in K:   # Cajitas completas: X_k = UC * C_k
    agregar("Empaque en cajitas", f"Cajitas_{corto[k]}", X[k], "=", UC * C[k], "und/sem")

# Horno: sum(TC_i * Y_i) <= H
agregar("Horno", "Disponibilidad_horno", pl.lpSum(TC[i] * Y[i] for i in I), "<=", H, "min/sem")

# Mano de obra: sum(TR_i * L_i) <= MO
agregar("Mano de obra", "Disponibilidad_mano_de_obra", pl.lpSum(TR[i] * L[i] for i in I), "<=", MO, "min-persona/sem")

print("Variables:", len(modelo.variables()), "| Restricciones:", len(modelo.constraints))

Variables: 27 | Restricciones: 29


## 6. Solución y estado

In [ ]:
modelo.solve(pl.PULP_CBC_CMD(msg=False))
estado = pl.LpStatus[modelo.status]
Z_optimo = pl.value(modelo.objective)
print("Estado:", estado, "| Z =", pesos(Z_optimo), "COP/semana")

Estado: Optimal | Z = $516.211,92 COP/semana


**Plan depurado.** Como hornear o preparar recetas de más no cambia la utilidad, el solucionador puede dejar horneadas vacías. Se resuelve un segundo paso con las mismas restricciones y la misma Z, que escoge el plan con **menos horneadas y recetas**.

In [ ]:
plan = pl.LpProblem("Plan_depurado", pl.LpMinimize)
plan += pl.lpSum(Y[i] + L[i] for i in I)
for nombre_r, restriccion in modelo.constraints.items():
    plan += restriccion, nombre_r
plan += pl.lpSum(U[i] * X[i] for i in I) >= Z_optimo - 0.01, "Mantener_Z_optima"

plan.solve(pl.PULP_CBC_CMD(msg=False))
print("Estado del plan:", pl.LpStatus[plan.status], "| Z =", pesos(pl.value(modelo.objective)), "COP/semana")

Estado del plan: Optimal | Z = $516.211,92 COP/semana


## 7. Resultados
### 7.1 Estado y función objetivo

In [ ]:
display(pd.DataFrame({"Resultado": ["Estado de la solución", "Utilidad máxima (Z)"],
                      "Valor": [estado, pesos(pl.value(modelo.objective)) + " COP/semana"]}))

,Resultado,Valor
0,Estado de la solución,Optimal
1,Utilidad máxima (Z),"$516.211,92 COP/semana"


### 7.2 Variables de decisión

In [ ]:
filas = []
for i in I:
    filas.append([nombre[i], int(D[i]), int(X[i].varValue), int(L[i].varValue), int(Y[i].varValue),
                  int(C[i].varValue) if i in K else "-", U[i] * X[i].varValue])

df_variables = pd.DataFrame(filas, columns=["Producto", "Demanda (und/sem)", "X: producir (und/sem)",
                                            "L: recetas/sem", "Y: horneadas/sem", "C: cajitas/sem",
                                            "Utilidad ($/sem)"], index=pd.Index(I, name="i"))
display(df_variables)

,Producto,Demanda (und/sem),X: producir (und/sem),L: recetas/sem,Y: horneadas/sem,C: cajitas/sem,Utilidad ($/sem)
i,,,,,,,
1,Pastel personal de zanahoria,4,4,2,1,-,"38,094.56"
2,Pastel personal forma de corazón,7,7,2,2,-,"61,679.10"
3,Cuchareable de maracumango,7,7,1,1,-,"32,031.30"
4,Cuchareable de fresa,12,10,1,1,-,"50,225.00"
5,Galleta de chocolate,68,68,1,3,34,"82,452.04"
6,Alfajor,64,64,2,3,32,"101,480.96"
7,Rollo de canela,68,68,3,7,34,"89,978.96"
8,Cuchareable de lulada,12,12,2,1,-,"60,270.00"


### 7.3 Restricciones: lado izquierdo, límite, holgura y estado

In [ ]:
filas = []
for grupo, nombre_r, lado_izq, sentido, limite, unidad in restricciones:
    li, lim = pl.value(lado_izq), pl.value(limite)
    holgura = 0.0 if abs(lim - li) < 1e-6 else lim - li
    filas.append([grupo, nombre_r, li, sentido, lim, holgura, unidad,
                  "Activa" if holgura == 0 else "No activa"])

df_restricciones = pd.DataFrame(filas, columns=["Grupo", "Restricción", "Lado izquierdo", "Sentido",
                                                "Límite", "Holgura", "Unidad", "Estado"])
display(df_restricciones)
print("Activas:", (df_restricciones["Estado"] == "Activa").sum(),
      "| No activas:", (df_restricciones["Estado"] == "No activa").sum())

,Grupo,Restricción,Lado izquierdo,Sentido,Límite,Holgura,Unidad,Estado
0,Demanda,Demanda_Zanahoria,4.00,<=,4.00,0.00,und/sem,Activa
1,Demanda,Demanda_Corazon,7.00,<=,7.00,0.00,und/sem,Activa
2,Demanda,Demanda_Maracumango,7.00,<=,7.00,0.00,und/sem,Activa
3,Demanda,Demanda_Fresa,10.00,<=,12.00,2.00,und/sem,No activa
4,Demanda,Demanda_Galleta,68.00,<=,68.00,0.00,und/sem,Activa
5,Demanda,Demanda_Alfajor,64.00,<=,64.00,0.00,und/sem,Activa
6,Demanda,Demanda_Rollo,68.00,<=,68.00,0.00,und/sem,Activa
7,Demanda,Demanda_Lulada,12.00,<=,12.00,0.00,und/sem,Activa
8,Capacidad por horneada,Capacidad_horneada_Zanahoria,4.00,<=,4.00,0.00,und/sem,Activa
9,Capacidad por horneada,Capacidad_horneada_Corazon,7.00,<=,8.00,1.00,und/sem,No activa


Activas: 13 | No activas: 16


## 8. Conclusiones

- **Utilidad máxima: \$516.211,92 COP/semana.** Se atiende toda la demanda de 7 productos; solo se prepara **1 genovesa de fresa** (10 de 12 vasos).
- **¿Por qué la fresa?** Cubrir toda la demanda exige 2.460 min-persona y hay 2.400, así que hay que dejar de hacer una receta. La segunda genovesa de fresa solo aporta 2 vasos (\$10.045), la menor pérdida posible. Recortar lulada da el mismo resultado (óptimo alternativo).
- **Mano de obra:** se usan 2.280 de 2.400 min-persona. Los 120 min libres no alcanzan para otra genovesa (180 min), por eso es el recurso que limita la producción.
- **Horno:** se usan 370 de 1.200 min (31 %); no es una restricción para la empresa.
- **Recetas:** rinden más de lo que se vende (sobran **86 unidades por semana**, por ejemplo, **26 galletas** y **36 alfajores**). Si no se venden, su costo no se recupera y la utilidad real sería menor que Z; si se venden aparte, son una oportunidad de venta adicional.

## 9. Análisis de sensibilidad

Como el modelo es **entero**, no se usan precios sombra. Cada escenario cambia **un solo parámetro**, se vuelve a resolver el modelo y se compara con la solución original.

La función `resolver` arma y resuelve el mismo modelo de las secciones 3 a 6 (incluido el plan depurado) con los parámetros que se le pasen.

In [ ]:
def resolver(D, U, MO=MO, H=H, continuo=False):
    m = pl.LpProblem("Escenario", pl.LpMaximize)
    dominio = pl.LpContinuous if continuo else pl.LpInteger
    x = {i: pl.LpVariable(f"X{i}", lowBound=0, cat=dominio) for i in I}
    y = {i: pl.LpVariable(f"Y{i}", lowBound=0, cat=dominio) for i in I}
    l = {i: pl.LpVariable(f"L{i}", lowBound=0, cat=dominio) for i in I}
    c = {k: pl.LpVariable(f"C{k}", lowBound=0, cat=dominio) for k in K}
    m += pl.lpSum(U[i] * x[i] for i in I)

    r = []  # (nombre, lado izquierdo, límite)
    for i in I:
        r.append((f"Demanda_{corto[i]}", x[i], D[i]))
        r.append((f"Capacidad_horneada_{corto[i]}", x[i], CH[i] * y[i]))
        r.append((f"Rendimiento_receta_{corto[i]}", x[i], RR[i] * l[i]))
    for nombre_r, li, lim in r:
        m += li <= lim, nombre_r
    for k in K:
        m += x[k] == UC * c[k], f"Cajitas_{corto[k]}"
        r.append((f"Cajitas_{corto[k]}", x[k], UC * c[k]))
    m += pl.lpSum(TC[i] * y[i] for i in I) <= H, "Disponibilidad_horno"
    m += pl.lpSum(TR[i] * l[i] for i in I) <= MO, "Disponibilidad_mano_de_obra"
    r.append(("Disponibilidad_horno", pl.lpSum(TC[i] * y[i] for i in I), H))
    r.append(("Disponibilidad_mano_de_obra", pl.lpSum(TR[i] * l[i] for i in I), MO))

    m.solve(pl.PULP_CBC_CMD(msg=False))
    estado = pl.LpStatus[m.status]
    if estado != "Optimal":
        raise RuntimeError(f"La optimización no fue óptima: {estado}")
    z = pl.value(m.objective)
    # Precios sombra únicamente de la RELAJACIÓN CONTINUA (nunca del MIP).
    precio_sombra_mo = m.constraints["Disponibilidad_mano_de_obra"].pi if continuo else None

    if not continuo:
        # Plan depurado: misma Z, mínimo de horneadas y recetas.
        p = pl.LpProblem("Plan", pl.LpMinimize)
        p += pl.lpSum(y[i] + l[i] for i in I)
        for nombre_r, restriccion in m.constraints.items():
            p += restriccion, nombre_r
        p += pl.lpSum(U[i] * x[i] for i in I) >= z - 0.01
        p.solve(pl.PULP_CBC_CMD(msg=False))
        if pl.LpStatus[p.status] != "Optimal":
            raise RuntimeError(f"Falló la depuración: {pl.LpStatus[p.status]}")
        if abs(pl.value(pl.lpSum(U[i] * x[i] for i in I)) - z) > 0.011:
            raise AssertionError("La depuración no conservó la utilidad óptima.")

    num = (lambda v: round(float(v), 6)) if continuo else (lambda v: int(round(float(v))))
    variables = pd.DataFrame({"X (und/sem)": {i: num(x[i].varValue) for i in I},
                              "L (recetas/sem)": {i: num(l[i].varValue) for i in I},
                              "Y (horneadas/sem)": {i: num(y[i].varValue) for i in I},
                              "C (cajitas/sem)": {i: num(c[i].varValue) if i in K else None for i in I}})
    filas = []
    for nombre_r, li, lim in r:
        h = pl.value(lim) - pl.value(li)
        h = 0.0 if abs(h) < 1e-6 else h
        filas.append([nombre_r, pl.value(li), pl.value(lim), h, "Activa" if h == 0 else "No activa"])
    restr = pd.DataFrame(filas, columns=["Restricción", "Lado izquierdo", "Límite", "Holgura", "Estado"])
    return {"estado": estado, "Z": z, "variables": variables, "restricciones": restr,
            "precio_sombra_mo": precio_sombra_mo}


def comparar(base, nuevo):
    # Función objetivo
    display(pd.DataFrame({"Original": [base["estado"], pesos(base["Z"])],
                          "Escenario": [nuevo["estado"], pesos(nuevo["Z"])],
                          "Cambio": ["", pesos(nuevo["Z"] - base["Z"])]},
                         index=["Estado", "Z (COP/semana)"]))
    # Variables por producto
    v = pd.concat({"Original": base["variables"], "Escenario": nuevo["variables"]}, axis=1)
    v.insert(0, ("", "Producto"), [nombre[i] for i in I])
    v.index.name = "i"
    display(v)
    # Restricciones que cambiaron de holgura o de estado
    rc = base["restricciones"].merge(nuevo["restricciones"], on="Restricción", suffixes=(" original", " escenario"))
    cambios = rc[(rc["Holgura original"] != rc["Holgura escenario"]) | (rc["Estado original"] != rc["Estado escenario"])]
    display(cambios[["Restricción", "Lado izquierdo original", "Lado izquierdo escenario", "Límite original",
                     "Límite escenario", "Holgura original", "Holgura escenario", "Estado original", "Estado escenario"]])


base = resolver(D, U)
print("Modelo original:", base["estado"], "| Z =", pesos(base["Z"]), "COP/semana")

Modelo original: Optimal | Z = $516.211,92 COP/semana


### 9.1 Escenario 1: aumento de la demanda del pastel de zanahoria (restricción activa)

- **Restricción:** `Demanda_Zanahoria` ($X_1 \leq D_1$), **activa** en la solución original.
- **Por qué es importante:** el pastel de zanahoria es el producto con **mayor utilidad por unidad** (\$9.523,64).
- **Cambio:** $D_1$ pasa de **4 a 6 pasteles/semana** (+2), por un aumento de pedidos en temporadas como Amor y Amistad, Día de la Madre o diciembre.

In [ ]:
D_esc1 = dict(D)
D_esc1[1] = 6          # demanda de pastel de zanahoria: 4 -> 6 und/sem

esc1 = resolver(D_esc1, U)
comparar(base, esc1)

,Original,Escenario,Cambio
Estado,Optimal,Optimal,
Z (COP/semana),"$516.211,92","$535.259,20","$19.047,28"


...       Escenario
                           Producto  ... C (cajitas/sem)
i                                    ...                
1      Pastel personal de zanahoria  ...             NaN
2  Pastel personal forma de corazón  ...             NaN
3        Cuchareable de maracumango  ...             NaN
4              Cuchareable de fresa  ...             NaN
5              Galleta de chocolate  ...           34.00
6                           Alfajor  ...           32.00
7                   Rollo de canela  ...           34.00
8             Cuchareable de lulada  ...             NaN

[8 rows x 9 columns]

,Restricción,Lado izquierdo original,Lado izquierdo escenario,Límite original,Límite escenario,Holgura original,Holgura escenario,Estado original,Estado escenario
1,Capacidad_horneada_Zanahoria,4.00,6.00,4.00,8.00,0.00,2.00,Activa,No activa
27,Disponibilidad_horno,370.00,390.00,"1,200.00","1,200.00",830.00,810.00,No activa,No activa
28,Disponibilidad_mano_de_obra,"2,280.00","2,400.00","2,400.00","2,400.00",120.00,0.00,No activa,Activa


**Resultado:**
- **Z sube \$19.047,28** (de \$516.211,92 a \$535.259,20): los 2 pasteles adicionales × \$9.523,64. **Es un beneficio para la empresa.**
- **Variables:** $X_1$ pasa de 4 a 6, $L_1$ de 2 a 3 recetas y $Y_1$ de 1 a 2 horneadas. Los demás productos no cambian (sigue 1 genovesa de fresa).
- **Restricciones:** la **mano de obra pasa a ser activa** (2.400 de 2.400 min-persona). La tercera receta de zanahoria usa justo los 120 min que antes sobraban. La capacidad de horneada de zanahoria deja de ser activa (2 horneadas con espacio para 8 y se hornean 6). El horno sube de 370 a 390 min y sigue con holgura.
- **Significado:** la empresa puede atender este aumento **sin recortar otro producto**, pero queda con la mano de obra al límite. Cualquier pedido adicional obligaría a dejar de producir algo más.

### 9.2 Escenario 2: nuevo precio del cuchareable de fresa (coeficiente de la función objetivo)

- **Coeficiente:** $U_4$, utilidad del cuchareable de fresa = precio de venta − costo = \$15.067,50 − \$10.045,00 = **\$5.022,50/und**.
- **Cambio:** se **redondea el precio de venta a \$16.000**, un precio comercial más fácil de manejar en redes y WhatsApp. El costo no cambia, así que $U_4$ = \$16.000 − \$10.045 = **\$5.955,00/und**.

In [ ]:
precio_fresa_original = 15067.50
costo_fresa = 10045.00
precio_fresa_nuevo = 16000.00

U_esc2 = dict(U)
U_esc2[4] = precio_fresa_nuevo - costo_fresa     # 5.022,50 -> 5.955,00 $/und
print("U4 original:", pesos(U[4]), "| U4 nuevo:", pesos(U_esc2[4]))

esc2 = resolver(D, U_esc2)
comparar(base, esc2)

U4 original: $5.022,50 | U4 nuevo: $5.955,00


,Original,Escenario,Cambio
Estado,Optimal,Optimal,
Z (COP/semana),"$516.211,92","$527.401,92","$11.190,00"


...       Escenario
                           Producto  ... C (cajitas/sem)
i                                    ...                
1      Pastel personal de zanahoria  ...             NaN
2  Pastel personal forma de corazón  ...             NaN
3        Cuchareable de maracumango  ...             NaN
4              Cuchareable de fresa  ...             NaN
5              Galleta de chocolate  ...           34.00
6                           Alfajor  ...           32.00
7                   Rollo de canela  ...           34.00
8             Cuchareable de lulada  ...             NaN

[8 rows x 9 columns]

,Restricción,Lado izquierdo original,Lado izquierdo escenario,Límite original,Límite escenario,Holgura original,Holgura escenario,Estado original,Estado escenario
9,Demanda_Fresa,10.00,12.00,12.00,12.00,2.00,0.00,No activa,Activa
10,Capacidad_horneada_Fresa,10.00,12.00,20.00,20.00,10.00,8.00,No activa,No activa
11,Rendimiento_receta_Fresa,10.00,12.00,10.00,20.00,0.00,8.00,Activa,No activa
21,Demanda_Lulada,12.00,10.00,12.00,12.00,0.00,2.00,Activa,No activa
22,Capacidad_horneada_Lulada,12.00,10.00,20.00,20.00,8.00,10.00,No activa,No activa
23,Rendimiento_receta_Lulada,12.00,10.00,20.00,10.00,8.00,0.00,No activa,Activa


**Resultado:**
- **Cambian la solución óptima y la función objetivo.** Z sube **\$11.190** (de \$516.211,92 a \$527.401,92).
- **Variables:** ahora se producen **12 vasos de fresa** (2 genovesas) y **10 de lulada** (1 genovesa). Antes era al revés.
- **Restricciones activas:** ahora están activas la demanda de fresa y el rendimiento de la receta de lulada; dejan de estarlo la demanda de lulada y el rendimiento de la receta de fresa. El uso de horno (370 min) y de mano de obra (2.280 min-persona) no cambia.
- **Razón matemática:** la mano de obra obliga a dejar de hacer una genovesa. En el modelo original, fresa y lulada empataban: recortar cualquiera costaba 2 × \$5.022,50 = \$10.045. Con el nuevo precio, recortar fresa costaría 2 × \$5.955 = \$11.910, así que el modelo recorta la **lulada**, que sigue costando \$10.045.
- **Razón empresarial:** con el precio más alto, la fresa se vuelve el cuchareable más rentable, y conviene atender toda su demanda y reducir el de menor margen.
- **Rango:** mientras $U_4 \leq$ \$5.022,50, la solución no cambia (se recorta la fresa) y solo cambia Z, en 10 × la variación de $U_4$. Si $U_4$ supera \$5.022,50, se recorta la lulada.

### 9.3 Corridas complementarias (citadas en el documento)

Esta sección reproduce las tablas adicionales del informe con **corridas reales de PuLP**.
En cada corrida se reconstruye el modelo a partir de `resolver`; los dos escenarios
obligatorios anteriores siguen siendo independientes entre sí.

#### 9.3.1 Demanda de zanahoria: $D_1$ de 4 a 10 unidades por semana

Cada fila cambia únicamente la demanda máxima de zanahoria y vuelve a optimizar el modelo entero.


In [ ]:
import math

resultados_d1 = []
z_anterior = None
for demanda_zanahoria in range(4, 11):
    d_prueba = dict(D)
    d_prueba[1] = demanda_zanahoria
    resultado = resolver(d_prueba, U)
    assert resultado["estado"] == "Optimal"
    v = resultado["variables"]
    trabajo_usado = sum(TR[i] * v.loc[i, "L (recetas/sem)"] for i in I)
    resultados_d1.append({
        "D1 (und/sem)": demanda_zanahoria,
        "X1 (und/sem)": v.loc[1, "X (und/sem)"],
        "L1 (recetas/sem)": v.loc[1, "L (recetas/sem)"],
        "Z (COP/sem)": pesos(resultado["Z"]),
        "ΔZ anterior (COP/sem)": "—" if z_anterior is None else pesos(resultado["Z"] - z_anterior),
        "Mano de obra usada (min-persona/sem)": trabajo_usado,
    })
    z_anterior = resultado["Z"]

df_rango_d1 = pd.DataFrame(resultados_d1)
display(df_rango_d1)


,D1 (und/sem),X1 (und/sem),L1 (recetas/sem),Z (COP/sem),ΔZ anterior (COP/sem),Mano de obra usada (min-persona/sem)
0,4,4,2,"$516.211,92",—,2280
1,5,5,3,"$525.735,56","$9.523,64",2400
2,6,6,3,"$535.259,20","$9.523,64",2400
3,7,6,3,"$535.259,20","$0,00",2400
4,8,8,4,"$544.261,48","$9.002,28",2340
5,9,8,4,"$544.261,48","$0,00",2340
6,10,10,5,"$545.686,16","$1.424,68",2340


#### 9.3.2 Utilidad unitaria de fresa: rango de $U_4$

Se modifica únicamente el coeficiente de la fresa. El precio equivalente se obtiene
sumando el costo constante de $10.045 COP por vaso. El resultado de $5.022,50 es un
**empate**: fresa y lulada ofrecen soluciones óptimas alternativas.


In [ ]:
valores_u4 = [3000.00, 4000.00, 4800.00, 5022.50, 5022.51, 5500.00, 5955.00, 7000.00]
resultados_u4 = []
for u4_nuevo in valores_u4:
    u_prueba = dict(U)
    u_prueba[4] = u4_nuevo
    resultado = resolver(D, u_prueba)
    assert resultado["estado"] == "Optimal"
    v = resultado["variables"]
    resultados_u4.append({
        "U4 (COP/und)": pesos(u4_nuevo),
        "Precio equivalente (COP/und)": pesos(costo_fresa + u4_nuevo),
        "Fresa X4 (und/sem)": v.loc[4, "X (und/sem)"],
        "Lulada X8 (und/sem)": v.loc[8, "X (und/sem)"],
        "Z (COP/sem)": pesos(resultado["Z"]),
    })

df_rango_u4 = pd.DataFrame(resultados_u4)
display(df_rango_u4)


,U4 (COP/und),Precio equivalente (COP/und),Fresa X4 (und/sem),Lulada X8 (und/sem),Z (COP/sem)
0,"$3.000,00","$13.045,00",10,12,"$495.986,92"
1,"$4.000,00","$14.045,00",10,12,"$505.986,92"
2,"$4.800,00","$14.845,00",10,12,"$513.986,92"
3,"$5.022,50","$15.067,50",10,12,"$516.211,92"
4,"$5.022,51","$15.067,51",12,10,"$516.212,04"
5,"$5.500,00","$15.545,00",12,10,"$521.941,92"
6,"$5.955,00","$16.000,00",12,10,"$527.401,92"
7,"$7.000,00","$17.045,00",12,10,"$539.941,92"


#### 9.3.3 Relajación lineal continua y precio sombra de mano de obra

Aquí se conserva **exactamente la misma función objetivo y las mismas 29 restricciones**,
pero se permite que `X`, `Y`, `L` y `C` sean continuas. Esto es un diagnóstico teórico:
una fracción de receta no representa una receta físicamente preparable. Su precio sombra
solo corresponde al **modelo relajado**, nunca al modelo original entero.


In [ ]:
relajacion = resolver(D, U, continuo=True)
assert relajacion["estado"] == "Optimal"
v_relajado = relajacion["variables"]
trabajo_relajado = sum(TR[i] * v_relajado.loc[i, "L (recetas/sem)"] for i in I)
df_relajacion = pd.DataFrame([
    {"Indicador": "Estado", "Resultado": relajacion["estado"], "Unidad": "—"},
    {"Indicador": "Utilidad óptima de la relajación", "Resultado": pesos(relajacion["Z"]), "Unidad": "COP/sem"},
    {"Indicador": "Mano de obra utilizada", "Resultado": round(trabajo_relajado, 2), "Unidad": "min-persona/sem"},
    {"Indicador": "Mano de obra disponible", "Resultado": MO, "Unidad": "min-persona/sem"},
    {"Indicador": "Holgura de mano de obra", "Resultado": round(MO - trabajo_relajado, 2), "Unidad": "min-persona/sem"},
    {"Indicador": "Precio sombra de mano de obra (solo LP)",
     "Resultado": round(relajacion["precio_sombra_mo"], 6), "Unidad": "COP/min-persona"},
])
display(df_relajacion)
print("NOTA: el valor dual mostrado no es un precio sombra del modelo entero.")


,Indicador,Resultado,Unidad
0,Estado,Optimal,—
1,Utilidad óptima de la relajación,"$526.256,92",COP/sem
2,Mano de obra utilizada,"2,400.00",min-persona/sem
3,Mano de obra disponible,2400,min-persona/sem
4,Holgura de mano de obra,-0.00,min-persona/sem
5,Precio sombra de mano de obra (solo LP),-0.00,COP/min-persona


NOTA: el valor dual mostrado no es un precio sombra del modelo entero.


#### 9.3.4 Variación de mano de obra en el modelo entero

Se cambia **únicamente** la disponibilidad de mano de obra `MO` y se vuelve a resolver:
2.280, 2.340, 2.400, 2.459, 2.460 y 2.520 min-persona por semana. El efecto es
escalonado debido a la indivisibilidad de las recetas.


In [ ]:
disponibilidades = [2280, 2340, 2400, 2459, 2460, 2520]
resultados_mo = []
for mo_prueba in disponibilidades:
    resultado = resolver(D, U, MO=mo_prueba)
    assert resultado["estado"] == "Optimal"
    v = resultado["variables"]
    trabajo_usado = sum(TR[i] * v.loc[i, "L (recetas/sem)"] for i in I)
    resultados_mo.append({
        "MO disponible (min-persona/sem)": mo_prueba,
        "MO utilizada (min-persona/sem)": trabajo_usado,
        "X4 fresa (und/sem)": v.loc[4, "X (und/sem)"],
        "X8 lulada (und/sem)": v.loc[8, "X (und/sem)"],
        "Z (COP/sem)": pesos(resultado["Z"]),
        "ΔZ vs. original (COP/sem)": pesos(resultado["Z"] - base["Z"]),
    })

df_variacion_mo = pd.DataFrame(resultados_mo)
display(df_variacion_mo)


,MO disponible (min-persona/sem),MO utilizada (min-persona/sem),X4 fresa (und/sem),X8 lulada (und/sem),Z (COP/sem),ΔZ vs. original (COP/sem)
0,2280,2280,10,12,"$516.211,92","$0,00"
1,2340,2280,10,12,"$516.211,92","$0,00"
2,2400,2280,10,12,"$516.211,92","$0,00"
3,2459,2280,10,12,"$516.211,92","$0,00"
4,2460,2460,12,12,"$526.256,92","$10.045,00"
5,2520,2460,12,12,"$526.256,92","$10.045,00"


#### 9.3.5 Validación: reglas hipotéticas A, B y C

Para comparar decisiones de recorte se parte de la demanda semanal completa. Como
harían falta 2.460 min-persona y solo hay 2.400, se calcula qué sucede si se obliga
a recortar **una receta completa de un producto seleccionado**, manteniendo todos los
demás parámetros constantes:

- **A.** Recortar galletas, el producto de menor utilidad unitaria.
- **B.** Recortar rollos, el producto que más mano de obra utiliza para cubrir su demanda.
- **C.** Elegir al azar, con la misma probabilidad, uno de los ocho productos para recortar:
  se calcula la pérdida **promedio teórica**, no una observación real.

En cada opción se fija un tope de ventas equivalente a prescindir de la última receta
del producto seleccionado y se resuelve el modelo entero. Las cajitas continúan siendo
de dos unidades. La diferencia de utilidad se mide frente al óptimo original.


In [ ]:
reglas_individuales = []
for i in I:
    recetas_demanda_completa = math.ceil(D[i] / RR[i])
    tope_con_una_receta_menos = min(D[i], (recetas_demanda_completa - 1) * RR[i])
    if i in K:
        tope_con_una_receta_menos -= tope_con_una_receta_menos % UC
    d_regla = dict(D)
    d_regla[i] = tope_con_una_receta_menos
    resultado = resolver(d_regla, U)
    assert resultado["estado"] == "Optimal"
    v = resultado["variables"]
    # Comprobar que no se recorten otros productos en esta comparación hipotética.
    assert all(v.loc[j, "X (und/sem)"] == d_regla[j] for j in I)
    reglas_individuales.append({
        "i": i,
        "Producto recortado": nombre[i],
        "X antes (und/sem)": D[i],
        "X después (und/sem)": tope_con_una_receta_menos,
        "Z de la regla (COP/sem)": resultado["Z"],
        "Pérdida frente al óptimo (COP/sem)": base["Z"] - resultado["Z"],
    })

df_reglas = pd.DataFrame(reglas_individuales).set_index("i")
display(df_reglas.assign(**{
    "Z de la regla (COP/sem)": df_reglas["Z de la regla (COP/sem)"].map(pesos),
    "Pérdida frente al óptimo (COP/sem)": df_reglas["Pérdida frente al óptimo (COP/sem)"].map(pesos),
}))

perdida_A = df_reglas.loc[5, "Pérdida frente al óptimo (COP/sem)"]
perdida_B = df_reglas.loc[7, "Pérdida frente al óptimo (COP/sem)"]
perdida_C = df_reglas["Pérdida frente al óptimo (COP/sem)"].mean()
df_resumen_reglas = pd.DataFrame([
    {"Regla": "A: recortar galletas", "Pérdida estimada": pesos(perdida_A),
     "Pérdida porcentual": f"{100 * perdida_A / base['Z']:.2f} %"},
    {"Regla": "B: recortar rollos", "Pérdida estimada": pesos(perdida_B),
     "Pérdida porcentual": f"{100 * perdida_B / base['Z']:.2f} %"},
    {"Regla": "C: elección aleatoria uniforme (promedio de 8)",
     "Pérdida estimada": pesos(perdida_C),
     "Pérdida porcentual": f"{100 * perdida_C / base['Z']:.2f} %"},
])
display(df_resumen_reglas)


,Producto recortado,X antes (und/sem),X después (und/sem),Z de la regla (COP/sem),Pérdida frente al óptimo (COP/sem)
i,,,,,
1,Pastel personal de zanahoria,4,2,"$507.209,64","$9.002,28"
2,Pastel personal forma de corazón,7,5,"$508.634,32","$7.577,60"
3,Cuchareable de maracumango,7,0,"$494.225,62","$21.986,30"
4,Cuchareable de fresa,12,10,"$516.211,92","$0,00"
5,Galleta de chocolate,68,0,"$443.804,88","$72.407,04"
6,Alfajor,64,50,"$504.057,96","$12.153,96"
7,Rollo de canela,68,52,"$505.085,40","$11.126,52"
8,Cuchareable de lulada,12,10,"$516.211,92","$0,00"


,Regla,Pérdida estimada,Pérdida porcentual
0,A: recortar galletas,"$72.407,04",14.03 %
1,B: recortar rollos,"$11.126,52",2.16 %
2,C: elección aleatoria uniforme (promedio de 8),"$16.781,71",3.25 %
